# Task : Build a RAG pipeline
#### in Phase 3 - Module 3: Retrieval-Augmented Generation (RAG)
----------

### Task 1: Load PDF content using free libraries

In [1]:
from pypdf import PdfReader

reader = PdfReader("./water_doc.pdf")
text = ""

for page in reader.pages:
    page_text = page.extract_text()
    
    if page_text:
        text += page_text + "\n"

print("Number of pages:", len(reader.pages))
print("\nFirst 260 characters:\n" , text[:260])

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)


Number of pages: 1

First 260 characters:
 Introduction 
Water management is the process of planning, developing, distributing, and managing the 
optimum use of water resources. It involves both water supply and water demand management. 
Section 1: Water Sources 
• Surface water (rivers, lakes, reservo


### Task 2: Split text into chunks

In [2]:
def split_text(text, chunk_size=500, overlap=50):
    chunks = []
    
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        
        if chunk:
            chunks.append(chunk)
        
        start += chunk_size - overlap
    return chunks


chunks = split_text(text, chunk_size=500, overlap=50)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i} :")
    print(chunk)

Number of chunks: 3

Chunk 0 :
Introduction 
Water management is the process of planning, developing, distributing, and managing the 
optimum use of water resources. It involves both water supply and water demand management. 
Section 1: Water Sources 
• Surface water (rivers, lakes, reservoirs) 
• Groundwater (wells, aquifers) 
• Rainwater harvesting 
Section 2: Water Distribution 
Water is distributed through pipelines, canals, and pumping systems to reach households, 
industries, and agriculture. Efficient distribution redu

Chunk 1 :
ries, and agriculture. Efficient distribution reduces water loss. 
Section 3: Water Quality Monitoring 
Monitoring water quality ensures safety and compliance with health standards. Key parameters 
include: 
• pH 
• Turbidity 
• Contaminant levels (heavy metals, chemicals) 
Section 4: Leak Detection 
Detecting leaks in pipelines is critical to minimize water loss. Techniques include: 
• Pressure monitoring 
• Acoustic sensors 
• Smart meters 
Conclusion

### Task 3: Generate embeddings for each chunk using sentence-transformers (all-MiniLM-L6-v2)

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(
    chunks,
    convert_to_numpy=True
)
print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", embeddings.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of embeddings: 3
Embedding dimension: 384


### Task 4: Store embeddings in memory or a free vector database (FAISS or Chroma, …)

In [4]:
import faiss

dimension = embeddings.shape[1]
print("Embedding dimension:", dimension)

index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))
print("Number of vectors in index:", index.ntotal)

Embedding dimension: 384
Number of vectors in index: 3


### Task 5: Implement semantic search to retrieve top 2–3 chunks based on user query

In [5]:
query = "What are the main sources of water?"
print("Query:", query)

query_embedding = model.encode(
    [query],
    convert_to_numpy=True
)

k = 3
distances, indices = index.search(
    query_embedding.astype("float32"), k)

for rank, idx in enumerate(indices[0], start=1):
    print(f"\nResult {rank} :")
    print(chunks[idx])
    print("Distance:", distances[0][rank - 1])

Query: What are the main sources of water?

Result 1 :
Introduction 
Water management is the process of planning, developing, distributing, and managing the 
optimum use of water resources. It involves both water supply and water demand management. 
Section 1: Water Sources 
• Surface water (rivers, lakes, reservoirs) 
• Groundwater (wells, aquifers) 
• Rainwater harvesting 
Section 2: Water Distribution 
Water is distributed through pipelines, canals, and pumping systems to reach households, 
industries, and agriculture. Efficient distribution redu
Distance: 1.0471799

Result 2 :
clusion 
Effective water management ensures sustainability and meets the growing demands of urban and 
rural populations. Combining monitoring, distribution, and leak detection improves efficiency.
Distance: 1.3239293

Result 3 :
ries, and agriculture. Efficient distribution reduces water loss. 
Section 3: Water Quality Monitoring 
Monitoring water quality ensures safety and compliance with health standards. 

### Task 6: Integrate retrieved chunks with an LLM like open ai

In [6]:
from pathlib import Path
from dotenv import load_dotenv
import os
from google import genai
from google.genai import types


env_path = Path.cwd()

while not (env_path / ".env").exists() and env_path != env_path.parent:
    env_path = env_path.parent

load_dotenv(env_path / ".env", override=True)
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("Put your Gemini key in .env as GEMINI_API_KEY=...")


client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(timeout=30_000)
)

### Task 7: Test pipeline with sample questions on the data

In [7]:
questions = [
    "What is water management?",
    "What are the main sources of water?",
    "How is water distributed?"
]

print("Number of test questions:", len(questions))

Number of test questions: 3


In [8]:
for query in questions:

    print("\n" + "=" * 70)
    print("QUESTION:", query)

    # Convert the question into an embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )


    # Retrieve top 3 relevant chunks
    k = 3
    distances, indices = index.search(
        query_embedding.astype("float32"),
        k
    )


    # Collect retrieved chunks
    retrieved_chunks = []
    for idx in indices[0]:

        retrieved_chunks.append(chunks[idx])


    # Combine retrieved chunks into one context
    context = "\n\n".join(retrieved_chunks)


    # Create prompt for the LLM

    prompt = f"""

Answer the question using only the information provided

in the context.


Context:

{context}


Question:

{query}

If the answer is not available in the context,

say that the information is not available in the document.

"""

    # Generate answer
    try:

        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )

        print("\nANSWER:")
        print(response.text)

    except Exception as e:

        print("\nGemini request failed:")
        print(type(e).__name__, e)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



QUESTION: What is water management?

ANSWER:
Based on the provided context, water management is the process of planning, developing, distributing, and managing the optimum use of water resources. It involves both water supply and water demand management.

QUESTION: What are the main sources of water?

ANSWER:
Based on the provided context, the water sources are:

* **Surface water** (rivers, lakes, reservoirs)
* **Groundwater** (wells, aquifers)
* **Rainwater harvesting**

QUESTION: How is water distributed?

ANSWER:
Based on the provided context, water is distributed through pipelines, canals, and pumping systems to reach households, industries, and agriculture.


### Task 8: Optimize chunk size, number of retrieved chunks, or embedding model for relevance

In [9]:
chunk_sizes = [300, 500, 700]
query = "What are the main sources of water?"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True
)

for size in chunk_sizes:

    print("\n" + "=" * 70)
    print("CHUNK SIZE:", size)

    # Create new chunks
    test_chunks = split_text(
        text,
        chunk_size=size,
        overlap=50
    )

    # Create embeddings for the new chunks
    test_embeddings = model.encode(
        test_chunks,
        convert_to_numpy=True
    )

    # Create a new FAISS index
    test_index = faiss.IndexFlatL2(
        test_embeddings.shape[1]
    )

    test_index.add(
        test_embeddings.astype("float32")
    )

    # Search for the query
    search_k = min(3, len(test_chunks))

    distances, indices = test_index.search(
        query_embedding.astype("float32"),
        search_k
    )

    # Display retrieved chunks
    for rank, idx in enumerate(indices[0], start=1):

        print(f"\nResult {rank}:")
        print(test_chunks[idx])
        print("Distance:", distances[0][rank - 1])


CHUNK SIZE: 300

Result 1:
s, reservoirs) 
• Groundwater (wells, aquifers) 
• Rainwater harvesting 
Section 2: Water Distribution 
Water is distributed through pipelines, canals, and pumping systems to reach households, 
industries, and agriculture. Efficient distribution reduces water loss. 
Section 3: Water Quality Monitori
Distance: 0.92352337

Result 2:
Introduction 
Water management is the process of planning, developing, distributing, and managing the 
optimum use of water resources. It involves both water supply and water demand management. 
Section 1: Water Sources 
• Surface water (rivers, lakes, reservoirs) 
• Groundwater (wells, aquifers) 
•
Distance: 1.0164542

Result 3:
ces water loss. 
Section 3: Water Quality Monitoring 
Monitoring water quality ensures safety and compliance with health standards. Key parameters 
include: 
• pH 
• Turbidity 
• Contaminant levels (heavy metals, chemicals) 
Section 4: Leak Detection 
Detecting leaks in pipelines is critical to mini
Distan

In [10]:
query = "What are the main sources of water?"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True
)

for k in [2, 3]:

    print("\n" + "=" * 70)
    print("TOP-K:", k)

    distances, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    for rank, idx in enumerate(indices[0], start=1):

        print(f"\nResult {rank}:")
        print(chunks[idx])
        print("Distance:", distances[0][rank - 1])


TOP-K: 2

Result 1:
Introduction 
Water management is the process of planning, developing, distributing, and managing the 
optimum use of water resources. It involves both water supply and water demand management. 
Section 1: Water Sources 
• Surface water (rivers, lakes, reservoirs) 
• Groundwater (wells, aquifers) 
• Rainwater harvesting 
Section 2: Water Distribution 
Water is distributed through pipelines, canals, and pumping systems to reach households, 
industries, and agriculture. Efficient distribution redu
Distance: 1.0471799

Result 2:
clusion 
Effective water management ensures sustainability and meets the growing demands of urban and 
rural populations. Combining monitoring, distribution, and leak detection improves efficiency.
Distance: 1.3239293

TOP-K: 3

Result 1:
Introduction 
Water management is the process of planning, developing, distributing, and managing the 
optimum use of water resources. It involves both water supply and water demand management. 
Section 1: Wa

### Note :
I understood the results by comparing the **question**, the **retrieved chunks**, and the **original PDF text**. I did not only look at the final LLM answer. I checked what FAISS retrieved before sending the context to the LLM.

For example, when the question was:

```text
What are the main sources of water?
```

The original PDF has this information in Section 1:

```text
Section 1: Water Sources
Surface water
Groundwater
Rainwater harvesting
```

The first retrieved chunk also contained the same information, so this means the semantic search retrieved the correct part of the PDF.

In Task 8, I compared the chunk sizes. When `chunk_size = 300`, the result started from the middle of the text:

```text
s, reservoirs)
Groundwater
Rainwater harvesting
```

This means the information was correct, but the chunk was a little small and cut the sentence.

When `chunk_size = 500`, the retrieved chunk was clearer:

```text
Introduction
Water management...
Section 1: Water Sources
Surface water...
Groundwater...
Rainwater harvesting
Section 2...
```

This chunk kept the related information together, so it was better for answering the question.

When `chunk_size = 700`, the answer information was still there, but the chunk also included extra information from other sections, like water quality monitoring. So it was bigger than needed.

The simple rule I used is:

```text
Best chunk = contains the answer + has less unrelated information + does not cut the text
```

From my results, chunk size 500 gave the best balance. Also, for this small PDF, top-k 2 is enough because it retrieves the main relevant chunks. Top-k 3 gives more context, but it may include extra information that is not needed.